In [ ]:
import numpy as np              # NumPy数值计算库
import matplotlib.pyplot as plt  # Matplotlib绑图库
import cv2 as cv                 # OpenCV计算机视觉库

In [ ]:
def show(img):
    """自定义显示函数：自动判断灰度图或彩色图并正确显示"""
    if img.ndim == 2:  # 灰度图
        plt.imshow(img, cmap='gray')
    else:  # 彩色图，BGR转RGB
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    plt.show()

## 1. 图像的裁剪、放大和缩小

In [ ]:
# 读取图片用于几何变换练习
img = cv.imread('pic/rabbit500x333.jpg')
show(img)

In [ ]:
# NumPy切片裁剪：提取第150-450行、第50-300列的区域（即兔子部分）
rabbit = img[150:450, 50:300, :]
show(rabbit)

In [ ]:
# cv.resize()：缩放图像，dsize参数为(宽度, 高度)，注意不是(行, 列)
img2 = cv.resize(img, (500, 400))
show(img2)

In [ ]:
# 使用最近邻插值缩放图像，相比默认的双线性插值会有锯齿效果
img3 = cv.resize(img, (500, 400), interpolation=cv.INTER_NEAREST)
show(img3)

## 2. 平移变换

In [ ]:
# 重新读取原图用于平移变换
img = cv.imread('pic/rabbit500x333.jpg')
show(img)

In [ ]:
# 定义平移仿射矩阵：向右平移100像素，向下平移50像素
M = np.array([
    [1, 0, 100],  # x方向平移100
    [0, 1, 50]    # y方向平移50
], dtype=np.float32)

In [ ]:
# cv.warpAffine()：应用仿射变换，dsize=(宽度, 高度)
img2 = cv.warpAffine(img, M, (333, 500))
show(img2)

## 3. 错切变换

In [ ]:
# 重新读取原图用于错切变换
img = cv.imread('pic/rabbit500x333.jpg')
show(img)

In [ ]:
# x方向错切：shx=0.2，图像向右倾斜
M = np.array([
    [1, 0.2, 0],  # x方向错切系数0.2
    [0, 1,   0]
], dtype=np.float32)

In [ ]:
# 应用x方向错切变换，输出尺寸调大以容纳变形后的图像
img3 = cv.warpAffine(img, M, (533, 500))
show(img3)

In [ ]:
# y方向错切：shy=0.3，图像向下倾斜
M = np.array([
    [1, 0, 0],
    [0.3, 1,   0]  # y方向错切系数0.3
], dtype=np.float32)

In [ ]:
# 应用y方向错切变换
img3 = cv.warpAffine(img, M, (333, 700))
show(img3)

## 4. 镜像变换

In [ ]:
# 重新读取原图用于镜像变换
img = cv.imread('pic/rabbit500x333.jpg')
show(img)

In [ ]:
# 水平镜像（左右翻转）：通过仿射矩阵实现
# x轴取反(-1)并平移图片宽度(333)，使翻转后的图像不超出画布
Mx = np.array([
    [-1, 0, 333],  # x轴翻转 + 平移
    [0,  1, 0]
], dtype=np.float32)

img2 = cv.warpAffine(img, Mx, (333, 500))
show(img2)

In [ ]:
# 垂直镜像（上下翻转）：y轴取反(-1)并平移图片高度(500)
My = np.array([
    [1, 0, 0],
    [0, -1, 500]  # y轴翻转 + 平移
], dtype=np.float32)

img3 = cv.warpAffine(img, My, (333, 500))
show(img3)

In [ ]:
# cv.flip()：更简洁的镜像函数
# flipCode: 1=水平翻转, 0=垂直翻转, -1=水平+垂直同时翻转
img4 = cv.flip(img, 1)   # 水平镜像
img5 = cv.flip(img, 0)   # 垂直镜像
img6 = cv.flip(img, -1)  # 水平和垂直同时进行

show(np.hstack([img, img4, img5, img6]))  # 原图 | 水平 | 垂直 | 两者

## 5. 旋转变换

In [ ]:
# 重新读取原图用于旋转变换
img = cv.imread('pic/rabbit500x333.jpg')
show(img)

In [ ]:
# 手动构建旋转矩阵：逆时针旋转45度（pi/4弧度）
# 旋转矩阵格式：[[cos, sin, 0], [-sin, cos, 0]]
beta = np.pi / 4  # 45度转弧度
M = np.array([
    [np.cos(beta), np.sin(beta), 0],
    [-np.sin(beta), np.cos(beta), 0]
], dtype=np.float32)

In [ ]:
# 应用手动构建的旋转矩阵
img2 = cv.warpAffine(img, M, (533, 500))
show(img2)

In [ ]:
# 获取图像的高度h、宽度w、通道数c
h, w, c = img.shape

In [ ]:
# cv.getRotationMatrix2D()：自动计算旋转矩阵
# 参数：(旋转中心x, y, 旋转角度度数, 缩放比例)
# 以图像中心为旋转点，旋转45度，不缩放
M2 = cv.getRotationMatrix2D((w//2, h//2), 45, 1)
M2

In [ ]:
# 以图像中心为旋转中心，旋转45度
img3 = cv.warpAffine(img, M2, (533, 500))
show(img3)

In [ ]:
# cv.rotate()：只能进行90度倍数的旋转，不支持任意角度
# ROTATE_90_CLOCKWISE：顺时针旋转90度
img4 = cv.rotate(img, cv.ROTATE_90_CLOCKWISE)
show(img4)

In [ ]:
# 实验一下把兔子图逆时针旋转90°


## 6. 透视变换

In [ ]:
# 读取帕特农神庙图片用于透视变换
img = cv.imread('pic/parthenon500x750.jpg')
show(img)

In [ ]:
# 透视变换的4个源点：原图中梯形的四个顶点坐标
src = np.array([
    [210, 50],
    [610, 270],
    [650, 470],
    [150, 450]
], dtype=np.float32)

In [ ]:
# 透视变换的4个目标点：将梯形校正为矩形
dst = np.array([
   [150, 50],
   [650, 50],
   [650, 470],
   [150, 470]
], dtype=np.float32)

In [ ]:
# cv.getPerspectiveTransform()：根据源点和目标点计算3x3透视变换矩阵
M = cv.getPerspectiveTransform(src, dst)
M

In [58]:
h, w, c = img.shape

In [ ]:
# cv.warpPerspective()：应用透视变换，将梯形校正为矩形
img2 = cv.warpPerspective(img, M, (w, h))
show(img2)

## 7. 插值比较

In [ ]:
# 读取小尺寸图片用于插值方法比较
img = cv.imread('pic/rabbit50x33.jpg')
show(img)

In [ ]:
# 比较最近邻插值和双线性插值的放大效果
img1 = cv.resize(img, (330, 500), interpolation=cv.INTER_NEAREST)  # 最近邻：速度快但有锯齿
img2 = cv.resize(img, (330, 500), interpolation=cv.INTER_LINEAR_EXACT)  # 双线性精确版：更平滑

show(np.hstack([img1, img2]))  # 左：最近邻，右：双线性